# Homework 2 (part a): Slope limiters for finite volumes in 1D

#### Notebook Submission Deadline: Wednesday 25 November 2026 (by 9pm)

#### Homework Written Defence: Thursday 26 November 2026 (at 1pm)

#### Credits: 25 points

## Name:

- When you finish, please submit your notebook via the link provided.

- This homework is individual. Please include your name in the notebook.

- Within a **single python notebook**, solve the following problem:

## Problem 1. Slope limiters for the advection equation

In class we solved the linear advection equation in conservation form,

$$\frac{\partial u}{\partial t} + \frac{\partial F(u)}{\partial x} = 0, \qquad F(u) = c\,u$$

with a finite-volume method: cell averages, ghost cells, a limited slope in each cell, an upwind Riemann solver, and the conservative update:

$$u^{n+1}_i = u^{n}_i + \frac{\Delta t}{\Delta x} \left(F^{n+1/2}_{i-1/2} - F^{n+1/2}_{i+1/2}\right)$$

The interface states are predicted to the half time from the slope in the cell,

$$u^{n+1/2}_{i+1/2,L} = u^{n}_{i} + \frac{\Delta x}{2} \left(1 - c\frac{\Delta t}{\Delta x}\right) \left(\frac{\Delta u}{\Delta x}\right)_{i}, \qquad u^{n+1/2}_{i+1/2,R} = u^{n}_{i+1} - \frac{\Delta x}{2} \left(1 + c\frac{\Delta t}{\Delta x}\right) \left(\frac{\Delta u}{\Delta x}\right)_{i+1}$$

exactly as in **nb14**, where the slope was limited with **minmod** and with **MC**.

In this problem you will add 4 more limiters and evaluate what each one does to the solution.

### Limiters:

Every limiter can be written as a single function $\psi(r)$ of the ratio of successive gradients,

$$r_i = \frac{\Delta u_{L}}{\Delta u_{R}} = \frac{u_i - u_{i-1}}{u_{i+1} - u_i}, \qquad\qquad \left(\frac{\Delta u}{\Delta x}\right)_i = \psi(r_i)\, \frac{u_{i+1} - u_i}{\Delta x}$$

so that one function controls the whole scheme. The two limiters you already know read, in this notation,

$$\psi_{\rm minmod}(r) = \max\left[0,\ \min\left(1,\ r\right)\right], \qquad\qquad \psi_{\rm MC}(r) = \max\left[0,\ \min\left(2r,\ \frac{1 + r}{2},\ 2\right)\right]$$

and the 4 you have to add are:

$$\psi_{\rm vanAlbada}(r) = \begin{cases} \dfrac{r^{2} + r}{r^{2} + 1} & r > 0\\[8pt] 0 & r \leq 0\end{cases} \qquad\qquad \psi_{\rm OSPRE}(r) = \begin{cases} \dfrac{3\left(r^{2} + r\right)}{2\left(r^{2} + r + 1\right)} & r > 0\\[8pt] 0 & r \leq 0\end{cases}$$

$$\psi_{\rm vanLeer}(r) = \frac{r + \left|r\right|}{1 + \left|r\right|}, \qquad\qquad \psi_{\rm UMIST}(r) = \max\left[0,\ \min\left(2r,\ \frac{1 + 3r}{4},\ \frac{3 + r}{4},\ 2\right)\right]$$

These are the 4 limiters commonly offered in hydrodynamic codes.

A limiter is second-order TVD when:

$$\psi(r) = 0 \;\; {\rm for} \;\; r \leq 0, \qquad \psi(1) = 1, \qquad 0 \leq \psi(r) \leq \min\left(2r,\ 2\right)$$

and the region allowed by the last condition is what we call the Sweby diagram.

##### References:

- Flux limiters: https://en.wikipedia.org/wiki/Flux_limiter

### Coding tasks:

#### Solver and limiters:

**(a)** Build the solver with **python functions or python classes** that stores the grid, the ghost cells and the solution, with methods to: set the initial condition, fill periodic boundary conditions, return $\Delta t$ from a CFL number, build the interface states from a limited slope, solve the Riemann problem by upwinding, and do the conservative update. The limiter must be an argument, so that one object runs every case. Use $c = 1$ on $x \in [0, 1]$ with two initial conditions:

$$u_{\rm hat}(x, 0) = \begin{cases} 1 & 1/3 \leq x \leq 2/3\\[3pt] 0 & {\rm otherwise}\end{cases} \qquad\qquad u_{\rm gauss}(x, 0) = e^{-100\left(x - 1/2\right)^{2}}$$

**(b)** Write functions for $\psi(r)$ for the 4 new limiters, and also for minmod and MC. Make a high-quality, labeled plot of the six curves for $r \in [-1, 4]$, together with the boundaries of the second-order TVD region (Sweby diagram), and mark the point $\psi(1) = 1$ that every second-order limiter must pass through. Then verify numerically that your $\psi$ form reproduces the minmod and MC slopes of nb14 on a random data vector. (**Hint:** you may need to guard the ratio $r$ where $u_{i+1} - u_i$ vanishes or is very small to avoid divisions by zero.)

#### Errors and convergence:

**(c)** With $n_x = 256$ and ${\rm CFL} = 0.7$, advect both initial conditions for 5 full periods with eight schemes: Godunov (zero slope), the unlimited centred slope, and the six limiters. Make high-quality, labeled figures per each initial condition comparing the final profiles with the exact solution. Report the minimum and the maximum of each solution in a table, and say which schemes stay inside the bounds of the initial data.

**(d)** Write a function that returns the error of a run against the exact solution in the $L_1$ and $L_2$ norms:

$$E_1 = \frac{1}{N}\sum_{i}\left|u_i - u^{\rm exact}_i\right|, \qquad\qquad E_2 = \left[\frac{1}{N}\sum_{i} \left(u_i - u^{\rm exact}_i\right)^{2}\right]^{1/2}$$

Report $E_1$ and $E_2$ for the eight schemes and the two profiles of point (c). Then repeat the runs for one period at $n_x = 32, 64, 128, 256, 512$ (and also $1024$, if possible), make labeled log-log plots of $E_1$ and $E_2$ against $n_x$ with reference lines of slope $-1$ and $-2$.

### Analysis questions:

**(e)** Answer in your own words the following questions, using your figures and results:

- Which limiter is the most accurate for the Gaussian, and which one for the top hat? Are they the same? Why?

- What does the unlimited centred slope do to the top hat, and what does Godunov's theorem indicate about it? How does the same scheme behave for the Gaussian? Why?

- How does your measured order of accuracy for the top hat compare to order $2$ for different limiters?

- Order and list the 6 limiters from the most diffusive to the most compressive, using your own numbers, and relate that order to where each curve sits inside the Sweby diagram of point (b).

- The van Albada and OSPRE limiters are smooth functions of $r$, while minmod, MC and UMIST are built from $\min$ and $\max$. What practical advantage does a smooth limiter have in a code that also computes derivatives of the solution?

### Animation:

**(f)** Create and display a GIF animation. Take the limiter that came out best for the top hat, advect it again for the 5 periods at $n_x = 256$, and save one frame every few steps as a GIF. Keep the exact profile in the frame as a reference, display the GIF in the notebook, and write one or two lines on what it shows. (**Note:** if the GIF makes your notebook too heavy to submit, comment out the line that saves it and leave the rest of the cell in place.)